# 🛡️ Risk Report – Quant Analyst
**Workflow**: Portfolio metrics → HRP weights → CVaR → Max Drawdown drill-down → Tail distribution → HTML export

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

session = AnalystSession(role=RoleContext.ANALYST)

SYMBOLS = ['BTC', 'ETH', 'SOL']
DAYS = 365
CONFIDENCE = 0.95  # VaR/CVaR confidence level

## 1. Load Portfolio Returns

In [ ]:
returns_dict = {}
for sym in SYMBOLS:
    try:
        df = session.load_ohlcv(sym, '1d')
    except Exception:
        df = session.sample_ohlcv(symbol=sym, days=DAYS)
    df = session.make_returns(df)
    returns_dict[sym] = df['returns'].drop_nulls().to_numpy()

# Align lengths
min_len = min(len(v) for v in returns_dict.values())
R = np.column_stack([v[-min_len:] for v in returns_dict.values()])
print(f"Returns matrix shape: {R.shape} ({min_len} periods x {len(SYMBOLS)} assets)")

## 2. Correlation Matrix

In [ ]:
corr = np.corrcoef(R.T)

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(SYMBOLS))); ax.set_xticklabels(SYMBOLS, color='#e2e8f0')
ax.set_yticks(range(len(SYMBOLS))); ax.set_yticklabels(SYMBOLS, color='#e2e8f0')
for i in range(len(SYMBOLS)):
    for j in range(len(SYMBOLS)):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', color='white', fontsize=11)
plt.colorbar(im, ax=ax)
ax.set_title('Correlation Matrix', color='#e2e8f0')
plt.tight_layout()
plt.show()
corr_fig = fig

## 3. VaR & CVaR (Historical Simulation)

In [ ]:
# Equal-weight portfolio
w = np.ones(len(SYMBOLS)) / len(SYMBOLS)
port_returns = R @ w

var_95 = float(np.percentile(port_returns, (1 - CONFIDENCE) * 100))
cvar_95 = float(port_returns[port_returns <= var_95].mean())

risk_metrics = {
    'VaR (95%)': var_95,
    'CVaR (95%)': cvar_95,
    'Annualised Vol': float(port_returns.std() * np.sqrt(252)),
    'Sharpe': float(port_returns.mean() / port_returns.std() * np.sqrt(252)) if port_returns.std() > 0 else 0,
    'Skewness': float(((port_returns - port_returns.mean())**3).mean() / (port_returns.std()**3 + 1e-10)),
    'Kurtosis': float(((port_returns - port_returns.mean())**4).mean() / (port_returns.std()**4 + 1e-10)) - 3,
}

risk_df = pl.DataFrame({'Metric': list(risk_metrics.keys()), 'Value': [round(v, 6) for v in risk_metrics.values()]})
risk_df

## 4. Return Distribution & Tail Risk

In [ ]:
fig2, ax = plt.subplots(figsize=(12, 4), facecolor='#0f1117')
ax.set_facecolor('#0f1117')
for sp in ax.spines.values(): sp.set_color('#334155')
ax.tick_params(colors='#94a3b8')
ax.grid(color='#1e293b', linestyle='--', linewidth=0.5)

ax.hist(port_returns, bins=80, color='#38bdf8', alpha=0.7, density=True)
ax.axvline(var_95, color='#f59e0b', linewidth=2, linestyle='--', label=f'VaR 95%: {var_95:.4f}')
ax.axvline(cvar_95, color='#ef4444', linewidth=2, linestyle='--', label=f'CVaR 95%: {cvar_95:.4f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1],
                  port_returns.min(), var_95, alpha=0.2, color='#ef4444', label='Tail')
ax.set_title('Portfolio Return Distribution & Tail Risk', color='#e2e8f0')
ax.legend(facecolor='#1e293b', labelcolor='#e2e8f0')
plt.tight_layout()
plt.show()
tail_fig = fig2

## 5. Export HTML Report

In [ ]:
out = session.export_report(
    title=f"Risk Report – {'/'.join(SYMBOLS)} Equal-Weight (95% CVaR)",
    sections={
        "Portfolio": f"Assets: {SYMBOLS} | Equal-weight | Confidence: {CONFIDENCE*100:.0f}%",
        "Risk Metrics": risk_metrics,
        "Correlation Matrix": corr_fig,
        "Return Distribution & Tail Risk": tail_fig,
    },
    path="reports/Risk_Report.html",
)
print(f"✅ Report saved: {out}")